# 05 · Machine Learning — Segmentation & Demand Forecast

Two ML applications. (Brief **§4.2 ML algorithms**, **§4.3 decision-making** — 20-mark section.)

1. **Customer segmentation (K-Means)** — cluster households by the *shape* of their daily
   load curve (from the half-hourly data) → behavioural segments.
2. **Demand forecasting (Gradient-Boosted Trees)** — predict per-home daily demand from
   weather + calendar → capacity planning.

> **Compute note:** Spark MLlib (`pyspark.ml`) is **not whitelisted on Serverless** compute.
> We therefore use the correct big-data pattern: **Spark does the distributed aggregation**
> (reducing 31.8M rows to a small table), then **scikit-learn** fits the model on the small
> aggregated result on the driver. This keeps the heavy lifting distributed while staying
> serverless-compatible.

In [ ]:
%run ./00_config_and_setup

In [ ]:
import sys, contextlib, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, mean_squared_error,
                             mean_absolute_error, r2_score)
from sklearn.ensemble import GradientBoostingRegressor

# On Databricks, scikit-learn's threadpoolctl helper fails to read a native library's
# version and raises an *unraisable* exception inside a ctypes callback, printing pages of
# harmless "Exception ignored ... 'NoneType' has no attribute 'split'". The model results
# are unaffected. We silence these at the source by no-op'ing the unraisable hook.
sys.unraisablehook = lambda unraisable: None


@contextlib.contextmanager
def quiet():
    """Belt-and-braces: also redirect stderr around the .fit() calls."""
    with contextlib.redirect_stderr(io.StringIO()):
        yield

## Part A — Household segmentation with K-Means
### A1. Build one average 48-point load curve per household (Spark), then L1-normalise
Spark pivots the half-hourly readings to a wide `hh_0..hh_47` profile (distributed over
31.8M rows → ~1,050 household rows). We then L1-normalise each row so clustering captures
the **shape** of usage (when energy is used), not its **size** (how much).

In [ ]:
hh_cols = [f"hh_{i}" for i in range(48)]

# Distributed aggregation in Spark: avg energy per (household, half-hour), pivoted wide
profiles_sdf = (spark.table(table("silver", "halfhourly"))
                .groupBy("LCLid")
                .pivot("half_hour", list(range(48)))
                .agg(F.avg("energy_kwh")))
for i in range(48):                                   # pivot names cols "0".."47"
    profiles_sdf = profiles_sdf.withColumnRenamed(str(i), f"hh_{i}")

# Collect the small result to the driver for scikit-learn
pdf = profiles_sdf.toPandas().fillna(0.0)
X = pdf[hh_cols].to_numpy()

# L1-normalise each household's curve (row sums to 1 -> share of daily energy per slot)
row_sums = X.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
Xn = X / row_sums
print(f"Households to cluster: {Xn.shape[0]:,}   (features: {Xn.shape[1]})")

### A2. Choose k via silhouette, then fit

In [ ]:
scores = {}
with quiet():
    for k in range(2, 7):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xn)
        scores[k] = silhouette_score(Xn, km.labels_)
for k in sorted(scores):
    print(f"k={k}  silhouette={scores[k]:.4f}")

# The silhouette is a flat plateau (~0.15) across k=2..4 — no single 'natural' k, and the
# differences are within KMeans' run-to-run (OpenMP) noise. Auto-selecting the max is
# unstable and can yield a degenerate tiny cluster (e.g. ~10 households at k=4). We
# therefore fix k=3, which gives balanced, interpretable load-shape segments.
best_k = 3
print(f"\nChosen k = {best_k}  (silhouette plateau; selected for balance & interpretability)")

In [ ]:
with quiet():
    km_model = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(Xn)
pdf["segment"] = km_model.labels_
print(pdf.groupby("segment").size().rename("households"))

### A3. Visualise the segment load shapes (the actionable output)

In [ ]:
centers = km_model.cluster_centers_
plt.figure(figsize=(11, 4))
for i, c in enumerate(centers):
    plt.plot(range(48), c, marker=".", label=f"Segment {i}")
plt.title("K-Means cluster centres — normalised daily load shape per segment")
plt.xlabel("Half-hour of day (0=00:00 … 47=23:30)"); plt.ylabel("Share of daily energy")
plt.xticks(range(0, 48, 4)); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
# SO WHAT: an "evening-peak" segment vs a "flat/daytime" segment lets the utility target
# each group with the right Time-of-Use tariff and demand-response programme.

In [ ]:
# Persist the household→segment assignment back to gold (Spark writes the Delta table)
seg_sdf = spark.createDataFrame(pdf[["LCLid", "segment"]])
seg_tbl = table("gold", "household_segments")
seg_sdf.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(seg_tbl)
print("saved ->", seg_tbl)

## Part B — Demand forecasting with Gradient-Boosted Trees
Target = **average daily kWh per home**; features = weather + calendar. We split by
**time** (train earlier, test later) — the honest way to evaluate a forecaster.

In [ ]:
# Small daily table -> bring to pandas for scikit-learn
tdf = (spark.table(table("gold", "daily_totals"))
       .select("day", "avg_kwh_per_home", "temp_avg", "season", "is_weekend", "is_holiday")
       .dropna(subset=["avg_kwh_per_home", "temp_avg"])
       .orderBy("day")
       .toPandas())

tdf["day"]         = pd.to_datetime(tdf["day"])
tdf["month"]       = tdf["day"].dt.month
tdf["day_of_week"] = tdf["day"].dt.dayofweek            # 0=Mon … 6=Sun
tdf["season_idx"]  = tdf["season"].astype("category").cat.codes

features = ["temp_avg", "month", "day_of_week", "is_weekend", "is_holiday", "season_idx"]

# Time-based hold-out: train on earliest 75% of days, test on the latest 25%
split = int(len(tdf) * 0.75)
train, test = tdf.iloc[:split], tdf.iloc[split:]
X_train, y_train = train[features], train["avg_kwh_per_home"]
X_test,  y_test  = test[features],  test["avg_kwh_per_home"]
print(f"split day={train['day'].max().date()}  train rows={len(train)}  test rows={len(test)}")

In [ ]:
gbt = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
with quiet():
    gbt.fit(X_train, y_train)
pred = gbt.predict(X_test)

### B1. Evaluate the forecast

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae  = mean_absolute_error(y_test, pred)
r2   = r2_score(y_test, pred)
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(test["day"], y_test.to_numpy(), label="Actual", lw=1)
plt.plot(test["day"], pred,             label="Predicted", lw=1)
plt.title("Demand forecast — actual vs predicted (held-out test period)")
plt.ylabel("kWh / home / day"); plt.legend(); plt.tight_layout(); plt.show()

### B2. Feature importance — what drives demand?

In [ ]:
for name, score in sorted(zip(features, gbt.feature_importances_),
                          key=lambda x: x[1], reverse=True):
    print(f"{name:<14} {score:.3f}")
# SO WHAT: temperature typically dominates -> weather-driven demand; the model lets the
# enterprise pre-position supply and size the grid for forecast peaks.